# 02 — Pré-processamento

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

RANDOM_STATE = 42

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path("..")
else:
    PROJECT_ROOT = Path(".")

RAW_APP = PROJECT_ROOT / "data" / "raw" / "application_record.csv"
RAW_CREDIT = PROJECT_ROOT / "data" / "raw" / "credit_record.csv"
PROCESSED = PROJECT_ROOT / "data" / "processed"
TARGET = "target"

pd.set_option("display.max_columns", None)

In [2]:
# Repetindo a lógica para criar o dataset base
df_app = pd.read_csv(RAW_APP)
df_credit = pd.read_csv(RAW_CREDIT)

bad_status = ['2', '3', '4', '5']
df_credit['is_bad'] = df_credit['STATUS'].isin(bad_status).astype(int)
target_df = df_credit.groupby('ID')['is_bad'].max().reset_index()
target_df.rename(columns={'is_bad': TARGET}, inplace=True)

df = df_app.merge(target_df, on='ID', how='inner')

## 1. Dados faltantes

In [3]:
nulos = df.isna().sum()
pd.DataFrame({"nulos": nulos, "pct": (nulos / len(df) * 100).round(2)}).query("nulos > 0")

,nulos,pct
OCCUPATION_TYPE,11323,31.06


In [4]:
# A principal variável com nulos é OCCUPATION_TYPE
# Preencher com "Desconhecido" é mais seguro do que descartar as linhas
df['OCCUPATION_TYPE'] = df['OCCUPATION_TYPE'].fillna('Unknown')

**Decisão:** _preencher._

## 2. Definição da variável alvo

_Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção._

In [5]:
# ex.: df['alvo'] = (df[TARGET] >= LIMIAR).astype(int)

## 3. Normalização / padronização

**Escolha e justificativa:** _StandardScaler, MinMaxScaler ou nenhum?_

Aplique preferencialmente dentro de um `Pipeline` no notebook 03 — ajustar o scaler antes do split vaza informação do teste para o treino.

In [6]:
# escala

## 4. Feature engineering

**Justificativa:** _preencher._

In [7]:
# novas variáveis derivadas

In [8]:
# 1. Transformar dias de vida em Idade em Anos
df['AGE_YEARS'] = (df['DAYS_BIRTH'] / -365.25).astype(int)

# 2. Tratar anomalia de DAYS_EMPLOYED (365243 = Desempregado/Pensionista)
df['FLAG_UNEMPLOYED'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['YEARS_EMPLOYED'] = np.where(df['DAYS_EMPLOYED'] == 365243, 0, df['DAYS_EMPLOYED'] / -365.25)

# 3. Remover colunas brutas e redundantes
cols_to_drop = ['DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
df = df.drop(columns=cols_to_drop)

# 4. One-Hot Encoding (Transformar variáveis categóricas em numéricas / dummies)
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

## 5. Salvar dataset tratado

In [9]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED / "dataset_tratado.csv", index=False)
print(f"Dataset salvo com shape: {df.shape}")

Dataset salvo com shape: (36457, 46)
